In [1]:
# ── Notebook 05 — TF-IDF Recommender ───────────────────────────────────────
# ── Cell 1 — Setup, load + build text field ─────────────────────────────────
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

REPO_ROOT  = Path().resolve().parents[1]
CLEAN_DIR  = REPO_ROOT / "Data" / "Clean"
MODEL_DIR  = REPO_ROOT / "Models"
MODEL_DIR.mkdir(exist_ok=True)

with open(CLEAN_DIR / "merged_non_western_fantasy.json") as f:
    df = pd.DataFrame(json.load(f))

# Clean titles
def clean_title(title):
    title = re.sub(r'\s*[\(\[](hardcover|paperback|kindle edition|ebook|mass market paperback|audio cd|audiobook|unknown binding|library binding|board book|large print)[\)\]]',
                   '', title, flags=re.IGNORECASE)
    return title.strip()

df["title"] = df["title"].apply(clean_title)

# Build text corpus
def build_text(row):
    parts = []
    if row.get("description"):
        parts.append(str(row["description"]))
    subjects = row.get("subjects", [])
    if isinstance(subjects, list) and subjects:
        parts.append(" ".join(str(s) for s in subjects))
    parts.append(str(row.get("title", "")) * 2)
    parts.append(str(row.get("author", "")) * 2)
    return " ".join(parts).strip()

df["text"] = df.apply(build_text, axis=1)

print(f"✅ Loaded {len(df):,} books")
print(f"   Empty text: {(df['text'].str.strip() == '').sum()}")
print(f"   Avg text length: {df['text'].str.len().mean():.0f} chars")

✅ Loaded 3,973 books
   Empty text: 0
   Avg text length: 908 chars


In [2]:
import re

def clean_title(title):
    title = re.sub(r'\s*[\(\[](hardcover|paperback|kindle edition|ebook|mass market paperback|audio cd|audiobook|unknown binding|library binding|board book|large print)[\)\]]',
                   '', title, flags=re.IGNORECASE)
    return title.strip()

print("✅ clean_title defined")

✅ clean_title defined


In [3]:
# ── Reload df first ─────────────────────────────────────────────────────────
import json, re
import pandas as pd
from pathlib import Path

REPO_ROOT  = Path().resolve().parents[1]
CLEAN_DIR  = REPO_ROOT / "Data" / "Clean"

with open(CLEAN_DIR / "merged_non_western_fantasy.json") as f:
    df = pd.DataFrame(json.load(f))

df["title"] = df["title"].apply(clean_title)

# Rebuild text field
def build_text(row):
    parts = []
    if row.get("description"):
        parts.append(str(row["description"]))
    subjects = row.get("subjects", [])
    if isinstance(subjects, list) and subjects:
        parts.append(" ".join(str(s) for s in subjects))
    parts.append(str(row.get("title", "")) * 2)
    parts.append(str(row.get("author", "")) * 2)
    return " ".join(parts).strip()

df["text"] = df.apply(build_text, axis=1)

df["subjects"] = df["subjects"].apply(lambda x: x if isinstance(x, list) else [])
df.to_json(CLEAN_DIR / "merged_non_western_fantasy.json", orient="records", indent=2, force_ascii=False)
df.to_csv(CLEAN_DIR / "merged_non_western_fantasy.csv", index=False)
print(f"✅ Ready — {len(df):,} books, text field built")

✅ Ready — 3,973 books, text field built


In [4]:
# ── Cell 2 — Build TF-IDF matrix ───────────────────────────────────────────
vectorizer = TfidfVectorizer(
    max_features=15000,    # top 15k words
    min_df=2,              # word must appear in at least 2 books
    max_df=0.85,           # ignore words in more than 85% of books
    ngram_range=(1, 2),    # unigrams and bigrams
    stop_words="english",
    sublinear_tf=True,     # dampen term frequency
)

tfidf_matrix = vectorizer.fit_transform(df["text"])

print(f"✅ TF-IDF matrix built")
print(f"   Shape: {tfidf_matrix.shape}")
print(f"   Books: {tfidf_matrix.shape[0]:,}")
print(f"   Features (words): {tfidf_matrix.shape[1]:,}")
print(f"   Matrix sparsity: {100 * (1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])):.1f}%")

✅ TF-IDF matrix built
   Shape: (3973, 15000)
   Books: 3,973
   Features (words): 15,000
   Matrix sparsity: 99.6%


3,995 = number of books (one row per book)
15,000 = number of unique words/bigrams in the vocabulary (we set max_features=15000)

So it's a giant table where each row is a book and each column is a word. The value in each cell is the TF-IDF score — how important that word is for that book.
99.6% sparse means most cells are 0 — which makes sense. A book about Yoruba mythology uses words like "orisha" and "magic" but doesn't use words like "samurai" or "aztec". Most words don't appear in most books, so most cells are empty.
This is normal and expected for text data — scikit-learn stores it efficiently as a sparse matrix so it doesn't actually take up much memory.

In [5]:
# ── Cell 3 — Obscurity boost + recommender functions ───────────────────────

def obscurity_boost(num_ratings, weight=0.3):
    num_ratings = num_ratings if pd.notna(num_ratings) else 0
    boost = 1 / np.log1p(num_ratings + 1)
    return min(boost, 1.0)  # cap at 1.0

df["obscurity_score"] = df["num_ratings"].apply(obscurity_boost)

print(f"── Obscurity score examples (capped) ──")
examples = [
    ("Very popular (500K ratings)",  500000),
    ("Popular (50K ratings)",         50000),
    ("Known (5K ratings)",             5000),
    ("Niche (500 ratings)",             500),
    ("Hidden gem (50 ratings)",          50),
    ("Unknown (0 ratings)",               0),
]
for label, n in examples:
    print(f"  {label:35s}: {obscurity_boost(n):.3f}")

def recommend(query_title=None, query_keywords=None, top_n=10, 
              obscurity_weight=0.3, min_ratings=0):
    """
    Two modes:
    - query_title:    find books similar to a given title
    - query_keywords: find books matching keywords
    
    obscurity_weight: 0 = pure similarity, 1 = pure obscurity
    min_ratings: filter out books with fewer than this many ratings
    """
    
    # ── Mode 1: Books like X ───────────────────────────────────────────────
    if query_title:
        matches = df[df["title"].str.contains(query_title, case=False, na=False)]
        if len(matches) == 0:
            print(f"❌ '{query_title}' not found in dataset")
            return None
        
        # Use first match
        idx        = matches.index[0]
        query_vec  = tfidf_matrix[idx]
        print(f"📖 Finding books similar to: {df.iloc[idx]['title']} — {df.iloc[idx]['author']}")

    # ── Mode 2: Keyword search ─────────────────────────────────────────────
    elif query_keywords:
        query_vec = vectorizer.transform([query_keywords])
        print(f"🔍 Finding books matching: '{query_keywords}'")
    
    else:
        print("❌ Provide either query_title or query_keywords")
        return None

    # ── Compute similarity ─────────────────────────────────────────────────
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # ── Apply obscurity boost ──────────────────────────────────────────────
    boosted_scores = (
        (1 - obscurity_weight) * sim_scores +
        obscurity_weight * df["obscurity_score"].values
    )

    # ── Filter + rank ──────────────────────────────────────────────────────
    results = df.copy()
    results["similarity"]     = sim_scores
    results["boosted_score"]  = boosted_scores

    # Exclude the query book itself
    if query_title and len(matches) > 0:
        results = results.drop(index=matches.index[0])

    # Apply min_ratings filter
    if min_ratings > 0:
        results = results[results["num_ratings"] >= min_ratings]

    # Sort by boosted score
    results = results.sort_values("boosted_score", ascending=False).head(top_n)

    return results[["title", "author", "avg_rating", "num_ratings", 
                     "similarity", "boosted_score", "source_tag", "description"]]

print(f"\n✅ Recommender ready!")

── Obscurity score examples (capped) ──
  Very popular (500K ratings)        : 0.076
  Popular (50K ratings)              : 0.092
  Known (5K ratings)                 : 0.117
  Niche (500 ratings)                : 0.161
  Hidden gem (50 ratings)            : 0.253
  Unknown (0 ratings)                : 1.000

✅ Recommender ready!


In [6]:
# ── Cell 4 — Test the recommender ──────────────────────────────────────────

# ── Test 1: Books like Children of Blood and Bone ──────────────────────────
print("=" * 60)
results1 = recommend("Children of Blood and Bone", top_n=10, obscurity_weight=0.3)
print(results1[["title", "author", "avg_rating", "num_ratings", "similarity"]].to_string())

# ── Test 2: Keyword search ─────────────────────────────────────────────────
print("\n" + "=" * 60)
results2 = recommend(query_keywords="japanese spirit world magic fox", top_n=10, obscurity_weight=0.3)
print(results2[["title", "author", "avg_rating", "num_ratings", "similarity"]].to_string())

# ── Test 3: Pure similarity (no obscurity boost) ───────────────────────────
print("\n" + "=" * 60)
print("Pure similarity (obscurity_weight=0):")
results3 = recommend("Children of Blood and Bone", top_n=10, obscurity_weight=0)
print(results3[["title", "author", "avg_rating", "num_ratings", "similarity"]].to_string())

📖 Finding books similar to: Children of Blood and Bone (Legacy of Orïsha, #1) — Tomi Adeyemi
                                                        title                 author  avg_rating  num_ratings  similarity
3193                         Righteous Blood, Ruthless Blades          Brendan Davis         NaN            0    0.068551
2     Children of Virtue and Vengeance (Legacy of Orïsha, #2)           Tomi Adeyemi        3.89        82391    0.447421
2181          Peculiar Ones : The Plantation of Windsor Ruins             Redsnapper         NaN            0    0.055557
2155                                        The Rain Princess            Katie Chase         NaN            0    0.053922
3884                                             Blood Spells       Jessica Andersen         NaN            0    0.053069
1121                                       House of the Beast          Michelle Wong         NaN            0    0.052556
2445                                  Anancy and Mr. 

Test 1: "Books like Children of Blood and Bone" (with obscurity boost)
The results are not great — "Righteous Blood Ruthless Blades", "Blood Spells", "Stan Lee's Convergence" are showing up because the obscurity boost is pushing unknown books (0 ratings) to the top regardless of similarity. The boost is too strong at weight=0.3.
Test 3: Pure similarity (no boost) — this is actually much better!

Children of Virtue and Vengeance ✅ (sequel, same series)
Kingdom of Souls ✅ (African fantasy)
Forest of Souls ✅ (similar vibes)
Blood Debts ✅ (African American fantasy)
Jade Legacy ✅ (Asian fantasy)

Test 2: Keyword search — mixed results. Some Japanese books showing up but also unrelated ones like "Unicorn Mountain" and "Islands in the Sky".
The diagnosis:

Pure similarity works well
Obscurity boost at 0.3 is too aggressive — it's surfacing random 0-rating books over genuinely similar ones
Keyword search needs a minimum similarity threshold

Let's tune the weights:

In [7]:
# ── Cell 5 — Tune the recommender ──────────────────────────────────────────

# Test with lower obscurity weight
print("── obscurity_weight=0.1 ──")
results = recommend("Children of Blood and Bone", top_n=10, obscurity_weight=0.1)
print(results[["title", "author", "avg_rating", "num_ratings", "similarity"]].to_string())

print("\n── obscurity_weight=0.15 ──")
results = recommend("Children of Blood and Bone", top_n=10, obscurity_weight=0.15)
print(results[["title", "author", "avg_rating", "num_ratings", "similarity"]].to_string())

── obscurity_weight=0.1 ──
📖 Finding books similar to: Children of Blood and Bone (Legacy of Orïsha, #1) — Tomi Adeyemi
                                                        title                 author  avg_rating  num_ratings  similarity
2     Children of Virtue and Vengeance (Legacy of Orïsha, #2)           Tomi Adeyemi        3.89        82391    0.447421
27     Children of Anguish and Anarchy (Legacy of Orïsha, #3)           Tomi Adeyemi        3.41        18754    0.400232
3193                         Righteous Blood, Ruthless Blades          Brendan Davis         NaN            0    0.068551
2181          Peculiar Ones : The Plantation of Windsor Ruins             Redsnapper         NaN            0    0.055557
2155                                        The Rain Princess            Katie Chase         NaN            0    0.053922
3884                                             Blood Spells       Jessica Andersen         NaN            0    0.053069
1121                      

In [8]:
# ── Cell 5 — Simplified recommender (pure similarity) ──────────────────────
def recommend(query_title=None, query_keywords=None, top_n=10, min_similarity=0.05):
    
    # ── Mode 1: Books like X ───────────────────────────────────────────────
    if query_title:
        matches = df[df["title"].str.contains(query_title, case=False, na=False)]
        if len(matches) == 0:
            print(f"❌ '{query_title}' not found in dataset")
            return None
        idx       = matches.index[0]
        query_vec = tfidf_matrix[idx]
        print(f"📖 Similar to: {df.iloc[idx]['title']} — {df.iloc[idx]['author']}")

    # ── Mode 2: Keyword search ─────────────────────────────────────────────
    elif query_keywords:
        query_vec = vectorizer.transform([query_keywords])
        print(f"🔍 Matching: '{query_keywords}'")
    else:
        print("❌ Provide either query_title or query_keywords")
        return None

    # ── Compute similarity ─────────────────────────────────────────────────
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()

    results = df.copy()
    results["similarity"] = sim_scores

    # Exclude query book
    if query_title and len(matches) > 0:
        results = results.drop(index=matches.index[0])

    # Filter by minimum similarity
    results = results[results["similarity"] >= min_similarity]

    # Sort by similarity
    results = results.sort_values("similarity", ascending=False).head(top_n)

    return results[["title", "author", "avg_rating", "num_ratings",
                     "similarity", "source_tag", "description"]]

# ── Test ───────────────────────────────────────────────────────────────────
print("=" * 60)
results1 = recommend("Children of Blood and Bone", top_n=10)
print(results1[["title", "author", "avg_rating", "num_ratings", "similarity"]].to_string())

print("\n" + "=" * 60)
results2 = recommend(query_keywords="japanese spirit world magic fox", top_n=10)
print(results2[["title", "author", "avg_rating", "num_ratings", "similarity"]].to_string())

📖 Similar to: Children of Blood and Bone (Legacy of Orïsha, #1) — Tomi Adeyemi
                                                        title                  author  avg_rating  num_ratings  similarity
2     Children of Virtue and Vengeance (Legacy of Orïsha, #2)            Tomi Adeyemi        3.89        82391    0.447421
27     Children of Anguish and Anarchy (Legacy of Orïsha, #3)            Tomi Adeyemi        3.41        18754    0.400232
63                    Kingdom of Souls (Kingdom of Souls, #1)             Rena Barron        3.72         5930    0.098017
326                          Forest of Souls (Shamanborn, #1)             Lori M. Lee        3.63         4530    0.090594
1003          We Shall Be Monsters (We Shall be Monsters, #1)                Tara Sim        3.87          970    0.084237
130                             Blood Debts (Blood Debts, #1)  Terry J. Benton-Walker        3.83         5029    0.083989
864                        The Starlight Heir (Starkeeper #1

In [9]:
# ── Cell 6 — Three-lane recommender ────────────────────────────────────────
def recommend_three_lanes(query_title, top_n=5):
    
    matches = df[df["title"].str.contains(query_title, case=False, na=False)]
    if len(matches) == 0:
        print(f"❌ '{query_title}' not found in dataset")
        return None

    idx        = matches.index[0]
    query_book = df.iloc[idx]
    query_vec  = tfidf_matrix[idx]
    query_author     = query_book["author"].lower().strip()
    query_source_tag = query_book["source_tag"]

    print(f"📖 Query: {query_book['title']} — {query_book['author']}")
    print(f"   Region: {query_source_tag}\n")

    # Compute similarity for all books
    sim_scores        = cosine_similarity(query_vec, tfidf_matrix).flatten()
    results           = df.copy()
    results["similarity"] = sim_scores
    results           = results.drop(index=idx)  # exclude query book

    # ── Lane 1: Same author / series ──────────────────────────────────────
    same_author = results[
        results["author"].str.lower().str.strip() == query_author
    ].sort_values("similarity", ascending=False).head(top_n)

    # ── Lane 2: Similar books, different author ────────────────────────────
    different_author = results[
        results["author"].str.lower().str.strip() != query_author
    ].sort_values("similarity", ascending=False).head(top_n)

    # ── Lane 3: Different region, still thematically related ──────────────
    different_region = results[
        (results["author"].str.lower().str.strip() != query_author) &
        (results["source_tag"] != query_source_tag)
    ].sort_values("similarity", ascending=False).head(top_n)

    # ── Print results ──────────────────────────────────────────────────────
    print("── Lane 1: More by this author ──")
    if len(same_author) > 0:
        print(same_author[["title", "author", "avg_rating", "num_ratings", "similarity"]].to_string())
    else:
        print("  No other books by this author in dataset")

    print("\n── Lane 2: Similar books ──")
    print(different_author[["title", "author", "avg_rating", "num_ratings", "similarity"]].to_string())

    print("\n── Lane 3: Discover something different ──")
    print(different_region[["title", "author", "avg_rating", "num_ratings", "similarity"]].to_string())

    return {"same_author": same_author, "similar": different_author, "discover": different_region}

# ── Test ───────────────────────────────────────────────────────────────────
results = recommend_three_lanes("Children of Blood and Bone", top_n=5)

📖 Query: Children of Blood and Bone (Legacy of Orïsha, #1) — Tomi Adeyemi
   Region: african-fantasy

── Lane 1: More by this author ──
                                                      title        author  avg_rating  num_ratings  similarity
2   Children of Virtue and Vengeance (Legacy of Orïsha, #2)  Tomi Adeyemi        3.89        82391    0.447421
27   Children of Anguish and Anarchy (Legacy of Orïsha, #3)  Tomi Adeyemi        3.41        18754    0.400232

── Lane 2: Similar books ──
                                                title                  author  avg_rating  num_ratings  similarity
63            Kingdom of Souls (Kingdom of Souls, #1)             Rena Barron        3.72         5930    0.098017
326                  Forest of Souls (Shamanborn, #1)             Lori M. Lee        3.63         4530    0.090594
1003  We Shall Be Monsters (We Shall be Monsters, #1)                Tara Sim        3.87          970    0.084237
130                     Blood Debts (Blood

In [10]:
# ── Cell 7 — Rebuilt Lane 3: Hidden Gems from underrepresented regions ──────

UNDERREPRESENTED = {
    "oceania", "australian-fantasy", "indigenous-fantasy", "indigenous_americas",
    "latin-american-fantasy", "latin_american", "south-american-fantasy",
    "middle-eastern-fantasy", "middle_eastern", "filipino", "southeast_asian",
    "african-science-fiction", "orisha", "igbo", "akan", "zulu", "yoruba",
    "anansi", "xianxia", "wuxia",
}

def recommend_three_lanes(query_title, top_n=5):

    matches = df[df["title"].str.contains(query_title, case=False, na=False)]
    if len(matches) == 0:
        print(f"❌ '{query_title}' not found in dataset")
        return None

    idx          = matches.index[0]
    query_book   = df.iloc[idx]
    query_vec    = tfidf_matrix[idx]
    query_author = query_book["author"].lower().strip()
    query_tag    = query_book["source_tag"]

    print(f"📖 Query: {query_book['title']} — {query_book['author']}")
    print(f"   Region tag: {query_tag}\n")

    sim_scores            = cosine_similarity(query_vec, tfidf_matrix).flatten()
    results               = df.copy()
    results["similarity"] = sim_scores
    results               = results.drop(index=idx)

    # ── Lane 1: Same author ────────────────────────────────────────────────
    same_author = results[
        results["author"].str.lower().str.strip() == query_author
    ].sort_values("similarity", ascending=False).head(top_n)

    # ── Lane 2: Similar books, different author ────────────────────────────
    similar = results[
        results["author"].str.lower().str.strip() != query_author
    ].sort_values("similarity", ascending=False).head(top_n)

    # ── Lane 3: Hidden gems from underrepresented regions ─────────────────
    # One best match per underrepresented region, min similarity 0.02
    hidden_gems_pool = results[
        (results["source_tag"].isin(UNDERREPRESENTED)) &
        (results["author"].str.lower().str.strip() != query_author) &
        (results["similarity"] >= 0.02)
    ]

    # Pick best match per region tag
    hidden_gems = (
        hidden_gems_pool
        .sort_values("similarity", ascending=False)
        .groupby("source_tag")
        .first()
        .reset_index()
        .sort_values("similarity", ascending=False)
        .head(top_n)
    )

    # ── Print results ──────────────────────────────────────────────────────
    print("── Lane 1: More by this author ──")
    if len(same_author) > 0:
        print(same_author[["title", "author", "avg_rating", "num_ratings", "similarity"]].to_string())
    else:
        print("  No other books by this author in dataset")

    print("\n── Lane 2: Similar books ──")
    print(similar[["title", "author", "avg_rating", "num_ratings", "similarity"]].to_string())

    print("\n── Lane 3: Hidden gems from underrepresented regions ──")
    if len(hidden_gems) > 0:
        print(hidden_gems[["title", "author", "avg_rating", "num_ratings", "similarity", "source_tag"]].to_string())
    else:
        print("  No hidden gems found")

    return {"same_author": same_author, "similar": similar, "hidden_gems": hidden_gems}

# ── Test ───────────────────────────────────────────────────────────────────
print("TEST 1:")
r1 = recommend_three_lanes("Children of Blood and Bone", top_n=5)

print("\n\nTEST 2:")
r2 = recommend_three_lanes("The Poppy War", top_n=5)

print("\n\nTEST 3:")
r3 = recommend_three_lanes("Three-Body Problem", top_n=5)

TEST 1:
📖 Query: Children of Blood and Bone (Legacy of Orïsha, #1) — Tomi Adeyemi
   Region tag: african-fantasy

── Lane 1: More by this author ──
                                                      title        author  avg_rating  num_ratings  similarity
2   Children of Virtue and Vengeance (Legacy of Orïsha, #2)  Tomi Adeyemi        3.89        82391    0.447421
27   Children of Anguish and Anarchy (Legacy of Orïsha, #3)  Tomi Adeyemi        3.41        18754    0.400232

── Lane 2: Similar books ──
                                                title                  author  avg_rating  num_ratings  similarity
63            Kingdom of Souls (Kingdom of Souls, #1)             Rena Barron        3.72         5930    0.098017
326                  Forest of Souls (Shamanborn, #1)             Lori M. Lee        3.63         4530    0.090594
1003  We Shall Be Monsters (We Shall be Monsters, #1)                Tara Sim        3.87          970    0.084237
130                     Blood 

In [11]:
# ── Cell 8 — Serialize model for Streamlit ─────────────────────────────────
import pickle
import numpy as np
from scipy import sparse

# Save TF-IDF matrix
sparse.save_npz(MODEL_DIR / "tfidf_matrix.npz", tfidf_matrix)

# Save vectorizer
with open(MODEL_DIR / "vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

# Save books index (df without text column)
df_save = df.drop(columns=["text"]).copy()
df_save.to_json(MODEL_DIR / "books_index.json", orient="records", indent=2, force_ascii=False)

print(f"✅ Model serialized to {MODEL_DIR}")
print(f"   tfidf_matrix.npz")
print(f"   vectorizer.pkl")
print(f"   books_index.json ({len(df_save):,} books)")

✅ Model serialized to C:\Users\Ready2Use\Documents\my-folder\Ironhack-week10\Book-recommendations\Models
   tfidf_matrix.npz
   vectorizer.pkl
   books_index.json (3,973 books)


In [12]:
from pathlib import Path

APP_DIR = Path().resolve().parents[1] / "App"
(APP_DIR / "pages").mkdir(parents=True, exist_ok=True)
(APP_DIR / "shared.py").touch()
(APP_DIR / "home.py").touch()
(APP_DIR / "pages" / "world_fantasy.py").touch()

print("✅ App structure created")
print("\n".join(str(p) for p in sorted(APP_DIR.rglob("*"))))

✅ App structure created
C:\Users\Ready2Use\Documents\my-folder\Ironhack-week10\Book-recommendations\App\home.py
C:\Users\Ready2Use\Documents\my-folder\Ironhack-week10\Book-recommendations\App\pages
C:\Users\Ready2Use\Documents\my-folder\Ironhack-week10\Book-recommendations\App\pages\world_fantasy.py
C:\Users\Ready2Use\Documents\my-folder\Ironhack-week10\Book-recommendations\App\shared.py


In [13]:
# ── Debug bio ─────────────────────────────────────────────────────────────
import requests
test = "Nnedi Okorafor"
r = requests.get(
    "https://en.wikipedia.org/api/rest_v1/page/summary/" + test.replace(" ", "_"),
    timeout=5
)
print(f"Status: {r.status_code}")
print(f"Type: {r.json().get('type')}")
print(f"Extract: {r.json().get('extract', '')[:200]}")
print(f"Image: {r.json().get('thumbnail', {}).get('source', 'none')}")

Status: 403


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
import requests
headers = {"User-Agent": "TheOtherShelf/1.0 (book recommender project)"}
r = requests.get(
    "https://en.wikipedia.org/api/rest_v1/page/summary/Nnedi_Okorafor",
    headers=headers,
    timeout=5
)
print(f"Status: {r.status_code}")
print(f"Text preview: {r.text[:200]}")

In [ ]:
# Run this once in your terminal or a notebook cell
import streamlit as st
st.cache_data.clear()